---
## 1. Setup & Imports


In [ ]:
# Colab setup
!pip -q install vllm sympy pandas tqdm

from pathlib import Path
from google.colab import drive
import os
import sys
import json
import gc
import re

import torch
import pandas as pd
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

from src.data.preprocess import load_eval_test_samples, format_as_prompt_completion
from src.eval.evaluate import compute_exact_match, compute_f1_for_task, is_answer_parsable, compute_grounding_rate

# Mount Drive
DRIVE_BASE = "/content/drive/MyDrive/FinReasoningAI"
drive.mount('/content/drive')

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
REPO_DIR = "/content/drive/MyDrive/FinReasoningAI/FinReasoningAI"
os.makedirs("/content/drive/MyDrive/FinReasoningAI", exist_ok=True)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo exists at {REPO_DIR}. Pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    print(f"Repo not found at {REPO_DIR}. Cloning...")
    !git clone {REPO_URL} {REPO_DIR}

PROJECT_DIR = REPO_DIR
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")

RAW_DATA_PATH = "/content/drive/MyDrive/FinReasoningAI/data/raw/synthetic.jsonl"
if not os.path.exists(RAW_DATA_PATH):
    RAW_DATA_PATH = "/content/drive/MyDrive/FinReasoningAI/data"

OUTPUTS_DIR = Path("/content/drive/MyDrive/FinReasoningAI/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/FinReasoningAI/outputs/sft_qlora/final_adapter"
MAX_SAMPLES = 200
GEN_MAX_TOKENS = 1024
GEN_TEMPERATURE = 0.0
GEN_TOP_P = 1.0

print("Setup complete.")
print(f"RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"ADAPTER_PATH: {ADAPTER_PATH}")


---
## 2. Mount Drive / Load Data


In [ ]:
# Use the same Step 2 split logic/format source as FinReasoningAI_Colab.ipynb
# (stratified split via src.data.preprocess, then evaluate on test split).

test_samples_all = load_eval_test_samples(
    data_path=RAW_DATA_PATH,
    train_frac=0.90,
    val_frac=0.05,
    seed=42,
)

print(f"Loaded test split size: {len(test_samples_all)}")

assert len(test_samples_all) > 0, "No test samples found. Check RAW_DATA_PATH."

eval_samples = test_samples_all[:MAX_SAMPLES]
print(f"Using {len(eval_samples)} samples for evaluation")

# Build exact Step 2-style prompts using format_as_prompt_completion
prompt_rows = []
for i, s in enumerate(eval_samples):
    pc = format_as_prompt_completion(s)
    prompt_rows.append({
        "idx": i,
        "id": s.get("id", str(i)),
        "task": s.get("task", "financial_qa"),
        "question": s.get("question", ""),
        "ground_truth": str(s.get("answer", "")),
        "context": str(s.get("context", "")),
        "expression": s.get("expression"),
        "variables": s.get("variables") if isinstance(s.get("variables"), dict) else {},
        "prompt": pc["prompt"],
    })

task_counts = {}
for r in prompt_rows:
    task_counts[r["task"]] = task_counts.get(r["task"], 0) + 1

print("Task distribution in eval slice:")
for task, n in sorted(task_counts.items()):
    print(f"- {task}: {n}")

print("\nPrompt preview:")
print(prompt_rows[0]["prompt"][:800])


---
## 3. Baseline Evaluation


In [ ]:
# Keep fine-tuned loading pattern (vLLM base + LoRA request), then run baseline with no adapter.

torch.cuda.empty_cache()
gc.collect()

sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

if "MODEL" not in globals() or MODEL is None:
    MODEL = LLM(
        model=MODEL_ID,
        enable_lora=True,
        max_lora_rank=64,
        gpu_memory_utilization=0.9,
        trust_remote_code=True,
        max_model_len=4096,
    )

sampling_params = SamplingParams(
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    max_tokens=GEN_MAX_TOKENS,
)

fin_lora = LoRARequest("fin_adapter", 1, ADAPTER_PATH)


def generate_texts(prompts, lora_request=None):
    outputs = MODEL.generate(prompts, sampling_params=sampling_params, lora_request=lora_request)
    return [o.outputs[0].text.strip() for o in outputs]


def score_rows(rows, pred_key):
    ems, f1s, pars, grs = [], [], [], []
    for r in rows:
        pred = r[pred_key]
        gt = r["ground_truth"]
        task = r["task"]

        em = compute_exact_match(pred, gt, tol=0.01)
        f1, _ = compute_f1_for_task(task, pred, gt, em)
        parsable = is_answer_parsable(pred)
        grounding = compute_grounding_rate(
            pred,
            r["context"],
            task=task,
            expression=str(r["expression"]) if r["expression"] is not None else None,
            variables=r["variables"],
            numeric_tolerance=0.01,
        )

        r[f"{pred_key}_exact_match"] = em
        r[f"{pred_key}_f1"] = f1
        r[f"{pred_key}_parsable"] = parsable
        r[f"{pred_key}_grounding_rate"] = grounding

        ems.append(em)
        f1s.append(f1)
        pars.append(float(parsable))
        grs.append(grounding)

    return {
        "exact_match": sum(ems) / len(ems),
        "f1": sum(f1s) / len(f1s),
        "parsability_rate": sum(pars) / len(pars),
        "grounding_rate": sum(grs) / len(grs),
        "n": len(rows),
    }

baseline_preds = generate_texts([r["prompt"] for r in prompt_rows], lora_request=None)

comparison_rows = []
for row, pred in zip(prompt_rows, baseline_preds):
    out = dict(row)
    out["baseline_prediction"] = pred
    comparison_rows.append(out)

baseline_metrics = score_rows(comparison_rows, "baseline_prediction")
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"- {k}: {v:.4f}" if isinstance(v, float) else f"- {k}: {v}")


---
## 4. Fine-tuned Model Evaluation


In [ ]:
# Fine-tuned pass (same base model + adapter loading logic via LoRARequest)

finetuned_preds = generate_texts([r["prompt"] for r in comparison_rows], lora_request=fin_lora)

for r, pred in zip(comparison_rows, finetuned_preds):
    r["finetuned_prediction"] = pred

finetuned_metrics = score_rows(comparison_rows, "finetuned_prediction")
print("Fine-tuned metrics:")
for k, v in finetuned_metrics.items():
    print(f"- {k}: {v:.4f}" if isinstance(v, float) else f"- {k}: {v}")


---
## 5. Results Comparison


In [ ]:
# Side-by-side comparison and persistence

summary = {
    "baseline": baseline_metrics,
    "finetuned": finetuned_metrics,
    "delta": {
        "exact_match": finetuned_metrics["exact_match"] - baseline_metrics["exact_match"],
        "f1": finetuned_metrics["f1"] - baseline_metrics["f1"],
        "parsability_rate": finetuned_metrics["parsability_rate"] - baseline_metrics["parsability_rate"],
        "grounding_rate": finetuned_metrics["grounding_rate"] - baseline_metrics["grounding_rate"],
    },
}

print("Comparison summary:")
for block, vals in summary.items():
    print(f"\n[{block}]")
    for k, v in vals.items():
        print(f"- {k}: {v:.4f}" if isinstance(v, float) else f"- {k}: {v}")

# By-task metrics
by_task = {}
for task in sorted({r["task"] for r in comparison_rows}):
    task_rows = [r for r in comparison_rows if r["task"] == task]
    b = {
        "em": sum(r["baseline_prediction_exact_match"] for r in task_rows) / len(task_rows),
        "f1": sum(r["baseline_prediction_f1"] for r in task_rows) / len(task_rows),
    }
    ft = {
        "em": sum(r["finetuned_prediction_exact_match"] for r in task_rows) / len(task_rows),
        "f1": sum(r["finetuned_prediction_f1"] for r in task_rows) / len(task_rows),
    }
    by_task[task] = {
        "baseline_em": b["em"],
        "finetuned_em": ft["em"],
        "delta_em": ft["em"] - b["em"],
        "baseline_f1": b["f1"],
        "finetuned_f1": ft["f1"],
        "delta_f1": ft["f1"] - b["f1"],
        "n": len(task_rows),
    }

print("\nBy-task comparison:")
for task, m in by_task.items():
    print(f"- {task}: n={m['n']}, EM {m['baseline_em']:.3f} -> {m['finetuned_em']:.3f} ({m['delta_em']:+.3f}), "
          f"F1 {m['baseline_f1']:.3f} -> {m['finetuned_f1']:.3f} ({m['delta_f1']:+.3f})")

results_path = OUTPUTS_DIR / "eval_baseline_vs_finetuned.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": summary,
        "by_task": by_task,
        "rows": comparison_rows,
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved comparison results to: {results_path}")


---
## 6. Export Full Predictions to CSV


In [ ]:
# Export full row-level comparison (test set + baseline + fine-tuned) to CSV

export_rows = []
for r in comparison_rows:
    export_rows.append({
        "idx": r.get("idx"),
        "id": r.get("id"),
        "task": r.get("task"),
        "question": r.get("question"),
        "context": r.get("context"),
        "expression": r.get("expression"),
        "variables": json.dumps(r.get("variables", {}), ensure_ascii=False),
        "ground_truth": r.get("ground_truth"),
        "baseline_prediction": r.get("baseline_prediction"),
        "finetuned_prediction": r.get("finetuned_prediction"),
        "baseline_exact_match": r.get("baseline_prediction_exact_match"),
        "finetuned_exact_match": r.get("finetuned_prediction_exact_match"),
        "baseline_f1": r.get("baseline_prediction_f1"),
        "finetuned_f1": r.get("finetuned_prediction_f1"),
        "baseline_parsable": r.get("baseline_prediction_parsable"),
        "finetuned_parsable": r.get("finetuned_prediction_parsable"),
        "baseline_grounding_rate": r.get("baseline_prediction_grounding_rate"),
        "finetuned_grounding_rate": r.get("finetuned_prediction_grounding_rate"),
    })

comparison_df = pd.DataFrame(export_rows)
csv_path = OUTPUTS_DIR / "eval_baseline_vs_finetuned_full.csv"
comparison_df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Saved CSV: {csv_path}")
print(f"Rows: {len(comparison_df)} | Cols: {len(comparison_df.columns)}")
comparison_df.head(3)
